In [1]:
!pip install wittgenstein -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.3/78.3 kB 4.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
import pandas as pd
import wittgenstein as lw

# Play Tennis dataset
data = {
    'Outlook': ['Sunny', 'Sunny', 'Overcast', 'Rain', 'Rain', 'Rain',
                'Overcast', 'Sunny', 'Sunny', 'Rain', 'Sunny', 'Overcast',
                'Overcast', 'Rain'],

    'Temperature': ['Hot', 'Hot', 'Hot', 'Mild', 'Cool', 'Cool',
                    'Cool', 'Mild', 'Cool', 'Mild', 'Mild', 'Mild',
                    'Hot', 'Mild'],

    'Humidity': ['High', 'High', 'High', 'High', 'Normal', 'Normal',
                 'Normal', 'High', 'Normal', 'Normal', 'Normal', 'High',
                 'Normal', 'High'],

    'Wind': ['Weak', 'Strong', 'Weak', 'Weak', 'Weak', 'Strong',
             'Strong', 'Weak', 'Weak', 'Weak', 'Strong', 'Strong',
             'Weak', 'Strong'],

    'PlayTennis': ['No', 'No', 'Yes', 'Yes', 'Yes', 'No',
                   'Yes', 'No', 'Yes', 'Yes', 'Yes', 'Yes',
                   'Yes', 'No']
}

df = pd.DataFrame(data)

print("Dataset:")
display(df)

Dataset:


,Outlook,Temperature,Humidity,Wind,PlayTennis
0,Sunny,Hot,High,Weak,No
1,Sunny,Hot,High,Strong,No
2,Overcast,Hot,High,Weak,Yes
3,Rain,Mild,High,Weak,Yes
4,Rain,Cool,Normal,Weak,Yes
5,Rain,Cool,Normal,Strong,No
6,Overcast,Cool,Normal,Strong,Yes
7,Sunny,Mild,High,Weak,No
8,Sunny,Cool,Normal,Weak,Yes
9,Rain,Mild,Normal,Weak,Yes


In [3]:
# Create RIPPER classifier
model = lw.RIPPER()

# Train the model
model.fit(df, class_feat='PlayTennis', pos_class='Yes')

print("RIPPER Classification Rules:")
print(model.out_model())

RIPPER Classification Rules:
[[Humidity=Normal] V
[Outlook=Overcast]]
None


In [5]:
# Predict the class for the dataset
predictions = model.predict(df)

print("Predictions:")
print(predictions)

Predictions:
[False, False, True, False, True, True, True, False, True, True, True, True, True, False]


In [7]:
from sklearn.metrics import accuracy_score

actual = df['PlayTennis']

# Map boolean predictions (True/False) to class labels ('Yes'/'No')
mapped_predictions = ['Yes' if p else 'No' for p in predictions]

accuracy = accuracy_score(actual, mapped_predictions)

print("Accuracy:", accuracy * 100, "%")

Accuracy: 85.71428571428571 %


In [9]:
# New weather condition
new_data = pd.DataFrame({
    'Outlook': ['Sunny'],
    'Temperature': ['Cool'],
    'Humidity': ['High'],
    'Wind': ['Strong']
})

prediction = model.predict(new_data)

print("New Example:")
display(new_data)

# Map the boolean prediction to the class name 'Yes' or 'No'
predicted_class = 'Yes' if prediction[0] else 'No'
print("Predicted Class:", predicted_class)

New Example:


,Outlook,Temperature,Humidity,Wind
0,Sunny,Cool,High,Strong


Predicted Class: No


In [10]:
# Facts about people

parent_facts = [
    ('John', 'Mary'),
    ('John', 'Bob'),
    ('Susan', 'Mary'),
    ('Susan', 'Bob'),
    ('Mary', 'Alice'),
    ('Bob', 'Tom')
]

female = ['Susan', 'Mary', 'Alice']

male = ['John', 'Bob', 'Tom']

print("Parent facts:")
for fact in parent_facts:
    print("parent(" + fact[0] + ", " + fact[1] + ")")

print("\nFemale:")
for person in female:
    print("female(" + person + ")")

Parent facts:
parent(John, Mary)
parent(John, Bob)
parent(Susan, Mary)
parent(Susan, Bob)
parent(Mary, Alice)
parent(Bob, Tom)

Female:
female(Susan)
female(Mary)
female(Alice)


In [11]:
# A mother is a female person who is also a parent

positive_examples = []

for parent in parent_facts:
    if parent[0] in female:
        positive_examples.append(parent)

print("Positive examples of mother:")
for x, y in positive_examples:
    print("mother(" + x + ", " + y + ")")

Positive examples of mother:
mother(Susan, Mary)
mother(Susan, Bob)
mother(Mary, Alice)


In [12]:
# Simple FOIL-style rule learning

def learn_mother_rule(parent_facts, female):

    print("Starting FOIL-style learning...\n")

    # Initial rule
    rule = "mother(X,Y) :- parent(X,Y)"

    print("Initial rule:")
    print(rule)

    # Check whether an additional condition is required
    print("\nGenerating candidate literals...")

    candidate = "female(X)"

    print("Candidate literal:", candidate)

    # Add the condition
    rule = "mother(X,Y) :- parent(X,Y), female(X)"

    print("\nFinal learned rule:")
    print(rule)

    return rule


final_rule = learn_mother_rule(parent_facts, female)

Starting FOIL-style learning...

Initial rule:
mother(X,Y) :- parent(X,Y)

Generating candidate literals...
Candidate literal: female(X)

Final learned rule:
mother(X,Y) :- parent(X,Y), female(X)


In [13]:
def predict_mother(person1, person2, parent_facts, female):

    if (person1, person2) in parent_facts and person1 in female:
        return True
    else:
        return False


print("Predictions using learned FOIL rule:\n")

test_cases = [
    ('Susan', 'Mary'),
    ('John', 'Mary'),
    ('Mary', 'Alice'),
    ('Bob', 'Tom')
]

for x, y in test_cases:
    result = predict_mother(x, y, parent_facts, female)

    print(f"mother({x}, {y}) -> {result}")

Predictions using learned FOIL rule:

mother(Susan, Mary) -> True
mother(John, Mary) -> False
mother(Mary, Alice) -> True
mother(Bob, Tom) -> False
